# RAG Monitoring Dashboard


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json, sqlite3, pandas as pd, matplotlib.pyplot as plt
import os
import httpx
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")

plt.style.use("seaborn-v0_8-whitegrid")
DB_PATH = Path.cwd().parent / "uploads/app_metadata.sqlite3"

PROJECT_ROOT = Path.cwd().parent if (Path.cwd().parent / "app").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")
DB_PATH = PROJECT_ROOT / "uploads/app_metadata.sqlite3"
MLRUNS_PATH = PROJECT_ROOT / "mlruns"


## Live System Metrics


In [ ]:
def table_exists(connection, table_name):
    row = connection.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
        (table_name,),
    ).fetchone()
    return row is not None

def sqlite_metrics_snapshot(db_path):
    if not db_path.exists():
        return {}
    with sqlite3.connect(db_path) as connection:
        if not table_exists(connection, "monitoring_metrics"):
            return {}
        df = pd.read_sql_query("SELECT metric_name, value FROM monitoring_metrics", connection)
    if df.empty:
        return {}
    grouped = df.groupby("metric_name")["value"]
    llm_tokens = df[df["metric_name"] == "llm_tokens"]["value"]
    return {
        "request_count": int((df["metric_name"] == "request_latency_ms").sum()),
        "average_request_latency_ms": grouped.mean().get("request_latency_ms", 0.0),
        "average_embedding_latency_ms": grouped.mean().get("embedding_latency_ms", 0.0),
        "average_reranking_latency_ms": grouped.mean().get("reranking_latency_ms", 0.0),
        "average_llm_latency_ms": grouped.mean().get("llm_latency_ms", 0.0),
        "average_qdrant_latency_ms": grouped.mean().get("qdrant_latency_ms", 0.0),
        "llm_average_tokens_per_request": float(llm_tokens.mean()) if not llm_tokens.empty else 0.0,
        "llm_total_tokens": int(llm_tokens.sum()) if not llm_tokens.empty else 0,
        "llm_request_count": int(len(llm_tokens)),
    }

api_key = os.environ.get("API_KEY", "")
headers = {"X-API-Key": api_key} if api_key else {}
try:
    response = httpx.get("http://localhost:8000/metrics", headers=headers, timeout=5.0)
    response.raise_for_status()
    metrics_snapshot = response.json()
    print("Loaded metrics from API.")
except Exception as exc:
    print(f"API unavailable; reading SQLite fallback. Reason: {exc}")
    metrics_snapshot = sqlite_metrics_snapshot(DB_PATH)

metrics_df = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in metrics_snapshot.items()]
)
display(metrics_df if not metrics_df.empty else pd.DataFrame(columns=["metric", "value"]))


## Request Latency Over Time


In [ ]:
def read_sqlite_query(query, params=None):
    if not DB_PATH.exists():
        return pd.DataFrame()
    with sqlite3.connect(DB_PATH) as connection:
        if "monitoring_metrics" in query and not table_exists(connection, "monitoring_metrics"):
            return pd.DataFrame()
        if "feedback" in query and not table_exists(connection, "feedback"):
            return pd.DataFrame()
        return pd.read_sql_query(query, connection, params=params or [])

latency_df = read_sqlite_query(
    "SELECT metric_name, value, created_at FROM monitoring_metrics "
    "WHERE metric_name = 'request_latency_ms' ORDER BY created_at"
)
if latency_df.empty or len(latency_df) < 5:
    print("Fewer than 5 request latency records exist; skipping chart.")
else:
    latency_df["created_at"] = pd.to_datetime(latency_df["created_at"], errors="coerce")
    latency_df = latency_df.dropna(subset=["created_at"])
    latency_df["rolling_avg"] = latency_df["value"].rolling(window=10, min_periods=1).mean()
    plt.figure(figsize=(12, 5))
    plt.plot(latency_df["created_at"], latency_df["value"], label="Latency")
    plt.plot(latency_df["created_at"], latency_df["rolling_avg"], label="10-sample average")
    plt.title("Request Latency Over Time")
    plt.ylabel("Latency (ms)")
    plt.xlabel("Time")
    plt.legend()
    plt.tight_layout()
    plt.show()


## Component Latency Breakdown


In [ ]:
latency_names = [
    "request_latency_ms",
    "embedding_latency_ms",
    "reranking_latency_ms",
    "llm_latency_ms",
    "qdrant_latency_ms",
]
placeholders = ",".join("?" for _ in latency_names)
breakdown_df = read_sqlite_query(
    f"SELECT metric_name, value FROM monitoring_metrics WHERE metric_name IN ({placeholders})",
    latency_names,
)
if breakdown_df.empty:
    print("No component latency records found.")
else:
    means = breakdown_df.groupby("metric_name")["value"].mean().sort_values(ascending=True)
    plt.figure(figsize=(12, 5))
    means.plot(kind="barh")
    plt.title("Average Latency by Component (ms)")
    plt.xlabel("Latency (ms)")
    plt.ylabel("Metric")
    plt.tight_layout()
    plt.show()


## LLM Token Usage


In [ ]:
tokens_df = read_sqlite_query(
    "SELECT value FROM monitoring_metrics WHERE metric_name = 'llm_tokens'"
)
if tokens_df.empty:
    print("No LLM token records found.")
else:
    token_values = tokens_df["value"]
    print(f"total tokens: {int(token_values.sum())}")
    print(f"avg per request: {token_values.mean():.2f}")
    print(f"max per request: {int(token_values.max())}")
    plt.figure(figsize=(12, 5))
    plt.hist(token_values, bins=30)
    plt.title("LLM Token Counts per Request")
    plt.xlabel("Tokens")
    plt.ylabel("Requests")
    plt.tight_layout()
    plt.show()


## User Feedback & Satisfaction


In [ ]:
feedback_df = read_sqlite_query(
    "SELECT rating, submitted_at, payload FROM feedback ORDER BY submitted_at"
)
if feedback_df.empty:
    print("No feedback records found.")
else:
    feedback_df["payload_json"] = feedback_df["payload"].apply(lambda value: json.loads(value) if isinstance(value, str) else {})
    feedback_df["submitted_at"] = pd.to_datetime(feedback_df["submitted_at"], errors="coerce")
    feedback_df["positive"] = (feedback_df["rating"] == 1).astype(int)
    feedback_df["rolling_satisfaction"] = feedback_df["positive"].rolling(window=10, min_periods=1).mean()
    positive_count = int((feedback_df["rating"] == 1).sum())
    negative_count = int((feedback_df["rating"] == -1).sum())
    total_count = int(len(feedback_df))
    satisfaction = positive_count / total_count if total_count else 0.0
    print(f"overall satisfaction score: {satisfaction:.3f}")
    print(f"total feedback count: {total_count}")
    print(f"positive count: {positive_count}")
    print(f"negative count: {negative_count}")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].bar(["positive", "negative"], [positive_count, negative_count], color=["green", "red"])
    axes[0].set_title("Feedback Counts")
    axes[0].set_ylabel("Count")
    axes[1].plot(feedback_df["submitted_at"], feedback_df["rolling_satisfaction"])
    axes[1].set_title("Rolling Satisfaction Score")
    axes[1].set_xlabel("Time")
    axes[1].set_ylabel("Score")
    axes[1].set_ylim(0, 1)
    plt.tight_layout()
    plt.show()


## MLflow Experiment Results


In [ ]:
try:
    import mlflow
    mlflow.set_tracking_uri(str(MLRUNS_PATH))
    runs = mlflow.search_runs(experiment_names=["RAG Pipeline Comparison"])
except Exception as exc:
    print(f"No MLflow comparison data available: {exc}")
    runs = pd.DataFrame()

metric_columns = [
    "metrics.faithfulness",
    "metrics.answer_relevancy",
    "metrics.context_recall",
    "metrics.context_precision",
]
if runs.empty:
    print("No RAG Pipeline Comparison runs exist yet.")
else:
    display_columns = ["tags.mlflow.runName"] + [column for column in metric_columns if column in runs.columns]
    comparison = runs[display_columns].copy()
    comparison = comparison.sort_values("metrics.faithfulness", ascending=False) if "metrics.faithfulness" in comparison else comparison
    best_run = comparison.iloc[0]["tags.mlflow.runName"] if not comparison.empty else "n/a"
    print(f"Best-performing run by faithfulness: {best_run}")
    display(comparison)
    plot_df = comparison.set_index("tags.mlflow.runName")[[column for column in metric_columns if column in comparison.columns]]
    if not plot_df.empty:
        plot_df.columns = [column.replace("metrics.", "") for column in plot_df.columns]
        plot_df.plot(kind="bar", figsize=(12, 5))
        plt.title("RAGAS Metrics by Run")
        plt.xlabel("Run")
        plt.ylabel("Score")
        plt.ylim(0, 1)
        plt.tight_layout()
        plt.show()


## Reindex Pipeline History


In [ ]:
try:
    import mlflow
    mlflow.set_tracking_uri(str(MLRUNS_PATH))
    reindex_runs = mlflow.search_runs(experiment_names=["Reindex Pipeline"])
except Exception as exc:
    print(f"No MLflow reindex data available: {exc}")
    reindex_runs = pd.DataFrame()

if reindex_runs.empty:
    print("No Reindex Pipeline runs exist yet.")
else:
    columns = [
        "start_time",
        "metrics.total_files",
        "metrics.reindexed_count",
        "metrics.failed_count",
        "metrics.duration_seconds",
    ]
    history = reindex_runs[[column for column in columns if column in reindex_runs.columns]].copy()
    history = history.sort_values("start_time") if "start_time" in history else history
    display(history)
    plot_history = history.set_index("start_time")[[
        column for column in ["metrics.reindexed_count", "metrics.failed_count"] if column in history.columns
    ]]
    if not plot_history.empty:
        plot_history.columns = ["reindexed", "failed"][: len(plot_history.columns)]
        plot_history.plot(kind="bar", figsize=(12, 5))
        plt.title("Reindex Results by Run")
        plt.xlabel("Run date")
        plt.ylabel("Files")
        plt.tight_layout()
        plt.show()
